# 02 — Exploratory Data Analysis
This notebook covers revenue, profit, orders, trends, products, customers, regions, salespeople, discounts, returns/cancellations, delivery performance, NumPy metrics, Pareto analysis, and RFM.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE=Path("..")
df=pd.read_csv(BASE/"data/processed/02_cleaned_sales_data.csv",parse_dates=["Order_Date"])
orders=df[df.Order_Flag.eq(1)].copy()


In [ ]:
total_revenue=df.Net_Sales_Amount.sum(); total_profit=df.Net_Profit.sum(); total_orders=df.Order_Flag.sum();
print({"Revenue":total_revenue,"Profit":total_profit,"Margin %":total_profit/total_revenue*100,"Orders":total_orders,"AOV":total_revenue/total_orders,"Net Quantity":df.Net_Quantity.sum()})


## Pandas operations used
`groupby()` and `agg()` build business summaries; `merge()` combines targets/master data; `pivot_table()` reshapes category-region results; `loc`/`iloc` select rows and columns; `apply`/`map`/`lambda` support row or label transformations; datetime and resampling create time analysis.


In [ ]:
monthly=df.set_index("Order_Date").resample("ME").agg(Revenue=("Net_Sales_Amount","sum"),Profit=("Net_Profit","sum")); monthly["MoM %"]=monthly.Revenue.pct_change()*100; monthly


In [ ]:
category_region=pd.pivot_table(df,index="Category",columns="Region",values="Net_Sales_Amount",aggfunc="sum",fill_value=0); category_region.head()


In [ ]:
top_products=df.groupby(["Product_ID","Product_Name"],as_index=False).agg(Revenue=("Net_Sales_Amount","sum"),Profit=("Net_Profit","sum")).sort_values("Revenue",ascending=False).head(10); top_products


In [ ]:
top_customers=df.groupby(["Customer_ID","Customer_Name"],as_index=False).agg(Revenue=("Net_Sales_Amount","sum"),Profit=("Net_Profit","sum"),Orders=("Order_Flag","sum")).sort_values("Revenue",ascending=False).head(10); top_customers


In [ ]:
# NumPy: distribution statistics and rule-based segmentation
arr=df.Net_Sales_Amount.to_numpy(); print(np.mean(arr),np.median(arr),np.std(arr),np.percentile(arr,[25,75,95]))
df["Discount_Band"]=np.select([df.Discount_Percent<=5,df.Discount_Percent<=10,df.Discount_Percent<=20],["0-5%","5-10%","10-20%"],default="20%+")


In [ ]:
discount=df.groupby("Discount_Band",as_index=False).agg(Revenue=("Net_Sales_Amount","sum"),Profit=("Net_Profit","sum")); discount["Margin %"]=discount.Profit/discount.Revenue*100; discount


In [ ]:
# Pareto
cust=df.groupby(["Customer_ID","Customer_Name"],as_index=False).Net_Sales_Amount.sum().sort_values("Net_Sales_Amount",ascending=False); cust["Cum Share"]=cust.Net_Sales_Amount.cumsum()/cust.Net_Sales_Amount.sum(); cust.head(20)


In [ ]:
# Rolling 3-month revenue average
monthly["Rolling_3M_Revenue"]=monthly.Revenue.rolling(3).mean(); monthly.tail()


In [ ]:
# Example Matplotlib chart
monthly.Revenue.plot(figsize=(10,4),title="Monthly Net Revenue"); plt.ylabel("INR"); plt.tight_layout(); plt.show()
